# 🌱🌡️ Descenso de Gradiente: prediciendo la altura de plantas en un invernadero

## El problema

Un agricultor tiene un invernadero con **30 eras** (camas de cultivo). En cada era controla la **temperatura promedio** (°C) y al final de la temporada mide la **altura promedio** de las plantas (cm).

| Era | Temperatura (°C) | Altura promedio (cm) |
|-----|------------------|---------------------|
| 1   | 15               | 42                  |
| 2   | 22               | 78                  |
| ... | ...              | ...                 |
| 30  | 35               | 110                 |

Quiere una **fórmula** para predecir la altura según la temperatura. La forma más sencilla: una **recta**.

## Regresión lineal: $y = mx + b$

Buscamos dos números:
- **m** (pendiente): cuántos cm más crece la planta por cada grado extra
- **b** (intercepto): altura base a temperatura mínima

## ¿Cómo saber si la recta es buena? La función de pérdida (MSE)

$$\text{MSE} = \frac{1}{N} \sum_{i=1}^{N} (y_{\text{pred},i} - y_{\text{real},i})^2$$

Mide la **distancia promedio al cuadrado** entre lo que predice la recta y la altura real medida. Si MSE = 0, la recta pasa por todos los puntos (imposible con variación natural).

## El descenso de gradiente: bajar la colina del error

Imagina que estás en lo alto de una **montaña con los ojos vendados**. ¿Cómo llegas al valle?

1. **Sientes la pendiente** bajo tus pies (el *gradiente*)
2. **Das un paso** en la dirección contraria a la pendiente (cuesta abajo)
3. **Repites** hasta llegar al fondo (mínimo del error)

El tamaño de cada paso es el **learning rate** ($\eta$).

## Los 3 tipos de descenso de gradiente

| Tipo | Datos por paso | Analogía del agricultor |
|------|---------------|-------------------------|
| **Batch GD** | TODOS los datos | Revisa las 30 eras antes de ajustar |
| **SGD** | UN dato aleatorio | Revisa UNA era al azar y ajusta |
| **Mini-batch GD** | Un GRUPO pequeño | Revisa un grupo de 8 eras y ajusta |

En este notebook implementaremos los 3 desde cero y los compararemos visualmente.

## 1. Importar librerías

Solo usamos **numpy** (cálculo numérico) y **matplotlib** (gráficas), ambas incluidas por defecto en Google Colab.

In [ ]:
import numpy as np                        # Álgebra lineal y operaciones numéricas
import matplotlib.pyplot as plt            # Gráficas 2D y 3D
from matplotlib import animation           # Animaciones cuadro por cuadro
from matplotlib import cm                  # Mapas de color para superficies
from IPython.display import HTML, display  # Mostrar el video HTML5 en Colab
import warnings
warnings.filterwarnings('ignore')          # Ocultar warnings cosméticos de matplotlib

print('Librerías cargadas correctamente')

## 2. 🌿 Datos del invernadero

Creamos datos de **30 eras** con:
- **X**: temperatura promedio de la era (°C, entre 12 y 38)
- **y**: altura promedio de las plantas (cm)

La relación real es $y \approx 3.2x + 5$ con ruido gaussiano, para que los datos no caigan perfectamente sobre una recta (como ocurre con la variabilidad natural).

Además, **normalizamos** X (restamos la media y dividimos por la desviación estándar) para que el descenso de gradiente converja bien y la superficie del MSE tenga forma **cónica simétrica** (contornos circulares en lugar de elípticos). Trabajaremos con X normalizado en todo el notebook.

In [ ]:
# ── Semilla para reproducibilidad ──
np.random.seed(42)

# ── Generar datos de temperatura por era ──
N = 30                                          # Número de eras en el invernadero
X_original = np.random.uniform(12, 38, N)       # Temperatura °C (entre 12 y 38)
X_original = np.sort(X_original)                # Ordenar para que se vea bonito

# ── Relación real: y = 3.2*x + 5 + ruido ──
# Cada grado extra aumenta ~3.2 cm de altura
ruido = np.random.normal(0, 8, N)               # Ruido gaussiano (media=0, std=8 cm)
y = 3.2 * X_original + 5 + ruido                # Altura promedio en cm

# ── Normalizar X (media=0, std=1) ──
# Esto hace que la superficie del MSE sea cónica (contornos circulares)
# en lugar de elíptica (contornos alargados), facilitando la convergencia
X_media = X_original.mean()                     # Media de X original
X_std = X_original.std()                        # Desviación estándar de X original
X = (X_original - X_media) / X_std              # X normalizado

# ── Visualizar los datos ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel izquierdo: datos originales
ax1.scatter(X_original, y, c='#4caf50', s=80, edgecolors='#2e7d32',
            lw=1.5, zorder=3, label='Eras')
for i in range(N):                              # Emoji de planta en cada punto
    ax1.text(X_original[i], y[i]+4, '\U0001F33F', ha='center', fontsize=8)
ax1.set_xlabel('Temperatura (°C)', fontsize=12)
ax1.set_ylabel('Altura promedio (cm)', fontsize=12)
ax1.set_title('\U0001F4CA Datos originales del invernadero', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=10)

# Panel derecho: datos normalizados (los que usaremos)
ax2.scatter(X, y, c='#2196f3', s=80, edgecolors='#1565c0',
            lw=1.5, zorder=3, label='Eras (normalizadas)')
for i in range(N):
    ax2.text(X[i], y[i]+4, '\U0001F33F', ha='center', fontsize=8)
ax2.set_xlabel('Temperatura normalizada', fontsize=12)
ax2.set_ylabel('Altura promedio (cm)', fontsize=12)
ax2.set_title('\U0001F4CA Datos normalizados (usaremos estos)', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

# ── Resumen ──
print(f'\n\U0001F3E1 Resumen de los datos del invernadero:')
print(f'  Eras: {N}')
print(f'  Temperatura: {X_original.min():.1f} - {X_original.max():.1f} °C')
print(f'  Altura:      {y.min():.1f} - {y.max():.1f} cm')
print(f'  X normalizado: media={X.mean():.4f}, std={X.std():.4f}')

## 3. 📏 La función de pérdida (MSE)

El **Error Cuadrático Medio** (Mean Squared Error) mide qué tan lejos está nuestra recta de los datos reales:

$$\text{MSE}(m, b) = \frac{1}{N} \sum_{i=1}^{N} \big((m \cdot x_i + b) - y_i\big)^2$$

- Si la recta pasa **lejos** de los puntos → MSE grande (❌)
- Si la recta pasa **cerca** de los puntos → MSE pequeño (✅)
- MSE = 0 significaría que la recta pasa por TODOS los puntos (imposible con variabilidad natural)

Podemos visualizar el MSE como una **superficie** en 3D: para cada combinación de (m, b), calculamos el error. Gracias a la normalización, el resultado tiene forma de **cono invertido** (contornos casi circulares). Nuestro objetivo: llegar al fondo del cono.

In [ ]:
def calcular_mse(X, y, m, b):
    """
    Calcula el Error Cuadrático Medio para una recta y = mx + b.

    Parámetros:
        X : array (N,) con los valores de entrada (normalizados)
        y : array (N,) con los valores reales
        m : float, pendiente de la recta
        b : float, intercepto de la recta

    Retorna:
        float : el MSE
    """
    y_pred = m * X + b                         # Predicción de la recta
    return np.mean((y_pred - y) ** 2)           # Promedio de errores al cuadrado


# ── Encontrar óptimos analíticamente para centrar la cuadrícula ──
# Con datos normalizados: m_opt ≈ correlación * std_y, b_opt ≈ mean_y
m_analitico = np.sum((X - X.mean()) * (y - y.mean())) / np.sum((X - X.mean())**2)
b_analitico = y.mean() - m_analitico * X.mean()

# ── Crear cuadrícula centrada en el óptimo con rango simétrico ──
rango_m = 60                                    # Rango simétrico alrededor del óptimo
rango_b = 60                                    # Mismo rango para b → contornos circulares
m_vals = np.linspace(m_analitico - rango_m, m_analitico + rango_m, 200)
b_vals = np.linspace(b_analitico - rango_b, b_analitico + rango_b, 200)
M, B = np.meshgrid(m_vals, b_vals)              # Cuadrícula 2D

# Calcular MSE en cada punto de la cuadrícula
Z = np.zeros_like(M)                            # Matriz para almacenar MSE
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        Z[i, j] = calcular_mse(X, y, M[i, j], B[i, j])

# ── Visualización: superficie 3D + contorno 2D ──
fig = plt.figure(figsize=(16, 6))

# Panel izquierdo: superficie 3D
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(M, B, Z, cmap=cm.viridis, alpha=0.8,
                 rstride=5, cstride=5, edgecolor='none')
ax1.set_xlabel('m (pendiente)', fontsize=11)
ax1.set_ylabel('b (intercepto)', fontsize=11)
ax1.set_zlabel('MSE', fontsize=11)
ax1.set_title('\U0001F3D4\uFE0F Superficie del error (MSE)', fontsize=13, fontweight='bold')
ax1.view_init(elev=35, azim=135)

# Panel derecho: contorno 2D (vista desde arriba)
ax2 = fig.add_subplot(122)
niveles = 20                                     # Número de líneas de contorno
cs = ax2.contour(M, B, Z, levels=niveles, cmap='viridis')
ax2.clabel(cs, fontsize=7, fmt='%.0f')          # Etiquetas de nivel
ax2.set_xlabel('m (pendiente)', fontsize=12)
ax2.set_ylabel('b (intercepto)', fontsize=12)
ax2.set_title('\U0001F5FA\uFE0F Mapa de contorno del MSE (forma cónica)', fontsize=13, fontweight='bold')

# Marcar el mínimo
ax2.plot(m_analitico, b_analitico, 'r*', markersize=20, zorder=5,
         label=f'Mínimo (m={m_analitico:.1f}, b={b_analitico:.1f})')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.2)
ax2.set_aspect('equal')                          # Aspecto 1:1 para ver la forma cónica

plt.tight_layout()
plt.show()

mse_optimo = calcular_mse(X, y, m_analitico, b_analitico)
print(f'\n\u2B50 El mínimo del MSE está en:')
print(f'   m = {m_analitico:.2f} (pendiente: cm por unidad de temperatura normalizada)')
print(f'   b = {b_analitico:.2f} (intercepto: altura base en cm)')
print(f'   MSE = {mse_optimo:.2f}')
print(f'\n\U0001F3AF Objetivo del descenso de gradiente: llegar al fondo del cono (la estrella roja)')
print(f'\n\U0001F4A1 Observa que los contornos son casi circulares gracias a la normalización.')
print(f'   Esto hace que el gradiente apunte directo al mínimo (convergencia rápida).')

## 4. ⤵\uFE0F ¿Cómo encontrar el mínimo? El descenso de gradiente

### Analogía: bajar una montaña con los ojos vendados

Estás en lo alto de la montaña del error. No puedes ver dónde está el valle, pero puedes **sentir la pendiente** bajo tus pies. Entonces:

1. **Sientes la pendiente** (calculas el gradiente)
2. **Das un paso cuesta abajo** (actualizas m y b)
3. **Repites** hasta que el suelo esté plano (el gradiente es ~0)

### Las fórmulas del gradiente

El gradiente del MSE respecto a m y b nos dice **en qué dirección sube** el error:

$$\frac{\partial \text{MSE}}{\partial m} = \frac{2}{N} \sum_{i=1}^{N} (m \cdot x_i + b - y_i) \cdot x_i$$

$$\frac{\partial \text{MSE}}{\partial b} = \frac{2}{N} \sum_{i=1}^{N} (m \cdot x_i + b - y_i)$$

### Regla de actualización

Nos movemos en la dirección **contraria** al gradiente (queremos bajar, no subir):

$$m \leftarrow m - \eta \cdot \frac{\partial \text{MSE}}{\partial m}$$
$$b \leftarrow b - \eta \cdot \frac{\partial \text{MSE}}{\partial b}$$

donde $\eta$ es el **learning rate** (tamaño del paso).

## 5. 📦 Tipo 1: Batch Gradient Descent

Usa **TODOS** los datos en cada paso para calcular el gradiente.

### Analogía
El agricultor recorre las **30 eras completas** del invernadero, anota las diferencias entre lo predicho y lo medido en todas, y **luego** ajusta su fórmula. Es el más preciso, pero el más lento si hay miles de datos.

### Ventajas y desventajas
- ✅ Dirección precisa (usa toda la información)
- ✅ Convergencia suave
- ❌ Lento si N es muy grande (millones de datos)

In [ ]:
def batch_gd(X, y, m_init, b_init, lr, epochs):
    """
    Batch Gradient Descent: usa TODOS los datos en cada paso.

    Parámetros:
        X       : array (N,) datos de entrada (normalizados)
        y       : array (N,) valores reales
        m_init  : float, pendiente inicial
        b_init  : float, intercepto inicial
        lr      : float, learning rate
        epochs  : int, número de épocas

    Retorna:
        historial : lista de dicts con m, b, mse en cada época
    """
    m = m_init                                  # Pendiente inicial
    b = b_init                                  # Intercepto inicial
    N = len(X)                                  # Número de datos
    historial = []                              # Guardar todo el progreso

    for epoca in range(epochs):
        # ── Paso 1: predicción actual ──
        y_pred = m * X + b                      # Predicción para TODOS los puntos

        # ── Paso 2: calcular el error ──
        error = y_pred - y                      # Diferencia (predicho - real)
        mse = np.mean(error ** 2)               # MSE actual

        # ── Paso 3: calcular gradientes (usando TODOS los datos) ──
        grad_m = (2 / N) * np.sum(error * X)    # ∂MSE/∂m
        grad_b = (2 / N) * np.sum(error)        # ∂MSE/∂b

        # ── Paso 4: actualizar parámetros (paso cuesta abajo) ──
        m = m - lr * grad_m                     # m nuevo = m viejo - η * gradiente_m
        b = b - lr * grad_b                     # b nuevo = b viejo - η * gradiente_b

        # ── Guardar para visualización ──
        historial.append({'m': m, 'b': b, 'mse': mse, 'epoca': epoca})

        # ── Imprimir progreso cada 10 épocas ──
        if epoca % 10 == 0 or epoca == epochs - 1:
            print(f'  Época {epoca:>3}: m={m:>8.3f}  b={b:>8.3f}  MSE={mse:>10.2f}')

    return historial


# ── Ejecutar Batch Gradient Descent ──
print('\U0001F4E6 BATCH GRADIENT DESCENT')
print('=' * 55)
print(f'  Parámetros iniciales: m=0, b=0 (recta y=0, muy mala)')
print(f'  Learning rate: 0.1')
print(f'  Épocas: 50')
print()

hist_batch = batch_gd(
    X, y,
    m_init = 0,                                 # Empezamos con pendiente = 0
    b_init = 0,                                 # Empezamos con intercepto = 0
    lr     = 0.1,                               # Learning rate
    epochs = 50                                  # 50 épocas
)

print(f'\n\u2705 Resultado final:')
print(f'   m = {hist_batch[-1]["m"]:.4f}')
print(f'   b = {hist_batch[-1]["b"]:.4f}')
print(f'   MSE = {hist_batch[-1]["mse"]:.4f}')

## 6. 🎬 Animación del Batch Gradient Descent

Video HTML5 mostrando:
- **Panel izquierdo**: los datos del invernadero y la recta ajustándose en cada época
- **Panel derecho**: la trayectoria de (m, b) bajando por el cono del MSE

In [ ]:
# ── Preparar datos para la animación ──
# Seleccionar frames: las primeras 15 épocas (donde pasan cosas) + cada 5 hasta el final
frames_idx = list(range(min(15, len(hist_batch))))
frames_idx += list(range(15, len(hist_batch), 5))
if len(hist_batch) - 1 not in frames_idx:       # Asegurar que el último frame esté
    frames_idx.append(len(hist_batch) - 1)
frames_idx = sorted(set(frames_idx))             # Eliminar duplicados y ordenar

# ── Crear figura ──
fig_anim, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(15, 6))
plt.close(fig_anim)

# Preparar contorno en panel derecho (fijo durante toda la animación)
m_range_anim = np.linspace(m_analitico - rango_m, m_analitico + rango_m, 150)
b_range_anim = np.linspace(b_analitico - rango_b, b_analitico + rango_b, 150)
Mg, Bg = np.meshgrid(m_range_anim, b_range_anim)
Zg = np.zeros_like(Mg)
for i in range(Mg.shape[0]):
    for j in range(Mg.shape[1]):
        Zg[i, j] = calcular_mse(X, y, Mg[i, j], Bg[i, j])


def animar_batch(frame_num):
    """Dibuja un frame de la animación del Batch GD."""
    idx = frames_idx[frame_num]                  # Índice real en el historial
    estado = hist_batch[idx]
    m_act = estado['m']
    b_act = estado['b']
    mse_act = estado['mse']

    # ── Panel izquierdo: datos + recta ──
    ax_left.clear()
    ax_left.scatter(X, y, c='#4caf50', s=70, edgecolors='#2e7d32',
                    lw=1.5, zorder=3)
    for i in range(N):                           # Emojis de planta
        ax_left.text(X[i], y[i]+3, '\U0001F33F', ha='center', fontsize=7)

    # Recta actual
    x_line = np.linspace(X.min()-0.5, X.max()+0.5, 100)
    y_line = m_act * x_line + b_act
    ax_left.plot(x_line, y_line, 'r-', lw=2.5, label=f'y = {m_act:.1f}x + {b_act:.1f}')

    ax_left.set_xlim(X.min()-0.5, X.max()+0.5)
    ax_left.set_ylim(y.min()-15, y.max()+15)
    ax_left.set_xlabel('Temperatura normalizada', fontsize=11)
    ax_left.set_ylabel('Altura (cm)', fontsize=11)
    ax_left.set_title(f'\U0001F4C8 Época {idx}: y = {m_act:.2f}x + {b_act:.2f}\nMSE = {mse_act:.2f}',
                      fontsize=12, fontweight='bold')
    ax_left.legend(fontsize=10, loc='upper left')
    ax_left.grid(True, alpha=0.3)

    # ── Panel derecho: contorno + trayectoria ──
    ax_right.clear()
    ax_right.contour(Mg, Bg, Zg, levels=20, cmap='viridis', alpha=0.7)
    ax_right.contourf(Mg, Bg, Zg, levels=20, cmap='viridis', alpha=0.3)

    # Trayectoria hasta este punto
    traj_m = [0] + [h['m'] for h in hist_batch[:idx+1]]
    traj_b = [0] + [h['b'] for h in hist_batch[:idx+1]]
    ax_right.plot(traj_m, traj_b, 'ro-', markersize=3, lw=1.5,
                 alpha=0.8, label='Trayectoria')
    ax_right.plot(m_act, b_act, 'r*', markersize=18, zorder=5,
                 label=f'Actual (m={m_act:.1f}, b={b_act:.1f})')
    ax_right.plot(m_analitico, b_analitico, 'g*', markersize=15, zorder=4,
                 label='\u00d3ptimo', alpha=0.6)

    ax_right.set_xlabel('m (pendiente)', fontsize=11)
    ax_right.set_ylabel('b (intercepto)', fontsize=11)
    ax_right.set_title(f'\U0001F5FA\uFE0F Trayectoria en el cono del error',
                       fontsize=12, fontweight='bold')
    ax_right.legend(fontsize=9, loc='upper right')
    ax_right.grid(True, alpha=0.2)
    ax_right.set_aspect('equal')

    fig_anim.suptitle('\U0001F4E6 Batch Gradient Descent', fontsize=14,
                      fontweight='bold', y=1.02)
    fig_anim.tight_layout()


# ── Crear y mostrar animación ──
anim_batch = animation.FuncAnimation(
    fig_anim,
    func     = animar_batch,
    frames   = len(frames_idx),
    interval = 2000,                             # 2 segundos por frame
    repeat   = True
)

display(HTML(f"""
<div style="text-align:center; margin:10px 0">
  <h3 style="color:#333">
    \U0001F4E6 Batch Gradient Descent: el agricultor revisa TODAS las eras
  </h3>
  <p style="color:#555; font-size:14px; max-width:700px; margin:8px auto">
    Observa cómo la recta (izquierda) se ajusta mientras el punto rojo (derecha)<br>
    baja suavemente por el cono del error hasta llegar al mínimo.
  </p>
  {anim_batch.to_html5_video()}
</div>
"""))

print('\n\U0001F441\uFE0F Qué observar:')
print('  1. La recta empieza plana (m=0, b=0) y se inclina hasta ajustarse a los datos')
print('  2. En el mapa de contorno, el punto baja en LÍNEA RECTA hacia el mínimo')
print('     (gracias a los contornos circulares del cono)')
print('  3. Los primeros pasos son grandes (pendiente empinada), luego se frena')

## 7. 🎲 Tipo 2: Stochastic Gradient Descent (SGD)

Usa **UN solo dato aleatorio** en cada paso para calcular el gradiente.

### Analogía
El agricultor elige **UNA era al azar** del invernadero, mira cuánto se equivocó su fórmula *solo ahí*, y ajusta. Es rápido pero **ruidoso**: cada era tira hacia un lado distinto, así que la trayectoria zigzaguea por el cono.

### Ventajas y desventajas
- ✅ Muy rápido por paso (solo procesa 1 dato)
- ✅ El ruido puede ayudar a escapar mínimos locales
- ❌ Trayectoria ruidosa (zigzaguea mucho)
- ❌ Necesita learning rate más pequeño para compensar el ruido

In [ ]:
def sgd(X, y, m_init, b_init, lr, epochs):
    """
    Stochastic Gradient Descent: usa UN dato aleatorio por paso.

    Parámetros:
        X       : array (N,) datos de entrada (normalizados)
        y       : array (N,) valores reales
        m_init  : float, pendiente inicial
        b_init  : float, intercepto inicial
        lr      : float, learning rate
        epochs  : int, número de épocas

    Retorna:
        historial : lista de dicts con m, b, mse en cada época
    """
    m = m_init                                  # Pendiente inicial
    b = b_init                                  # Intercepto inicial
    N = len(X)                                  # Número de datos
    historial = []                              # Guardar todo el progreso
    np.random.seed(42)                          # Reproducibilidad

    for epoca in range(epochs):
        # ── Mezclar los datos al inicio de cada época ──
        indices = np.random.permutation(N)      # Orden aleatorio

        for idx in indices:
            # ── Paso 1: predicción para UN solo punto ──
            y_pred_i = m * X[idx] + b            # Predicción para el punto idx

            # ── Paso 2: calcular error de ese punto ──
            error_i = y_pred_i - y[idx]          # Error individual

            # ── Paso 3: gradiente basado en UN solo punto ──
            grad_m = 2 * error_i * X[idx]        # ∂MSE/∂m (un solo dato)
            grad_b = 2 * error_i                 # ∂MSE/∂b (un solo dato)

            # ── Paso 4: actualizar parámetros ──
            m = m - lr * grad_m                  # Paso cuesta abajo en m
            b = b - lr * grad_b                  # Paso cuesta abajo en b

        # ── Al final de cada época, calcular MSE completo ──
        mse = calcular_mse(X, y, m, b)
        historial.append({'m': m, 'b': b, 'mse': mse, 'epoca': epoca})

        # ── Imprimir progreso ──
        if epoca % 10 == 0 or epoca == epochs - 1:
            print(f'  Época {epoca:>3}: m={m:>8.3f}  b={b:>8.3f}  MSE={mse:>10.2f}')

    return historial


# ── Ejecutar SGD ──
print('\U0001F3B2 STOCHASTIC GRADIENT DESCENT (SGD)')
print('=' * 55)
print(f'  Parámetros iniciales: m=0, b=0 (mismos que batch)')
print(f'  Learning rate: 0.01 (más pequeño por el ruido)')
print(f'  Épocas: 50')
print()

hist_sgd = sgd(
    X, y,
    m_init = 0,                                 # Mismos parámetros iniciales
    b_init = 0,
    lr     = 0.01,                              # Learning rate más pequeño
    epochs = 50
)

print(f'\n\u2705 Resultado final:')
print(f'   m = {hist_sgd[-1]["m"]:.4f}')
print(f'   b = {hist_sgd[-1]["b"]:.4f}')
print(f'   MSE = {hist_sgd[-1]["mse"]:.4f}')

## 8. 📦📦 Tipo 3: Mini-Batch Gradient Descent

Usa un **GRUPO pequeño** de datos (ej: 8 eras) en cada paso.

### Analogía
El agricultor revisa un **grupo de 8 eras** a la vez dentro del invernadero, calcula el error promedio de ese grupo, y ajusta. No son todas (como batch) ni una sola (como SGD), sino el **punto medio**.

### Ventajas y desventajas
- ✅ Más rápido que batch (no necesita todos los datos)
- ✅ Más estable que SGD (promedia varios puntos)
- ✅ **Es el método estándar en Deep Learning** (con batch_size = 32, 64, etc.)
- ❌ Hay que elegir el batch_size (hiperparámetro extra)

In [ ]:
def minibatch_gd(X, y, m_init, b_init, lr, epochs, batch_size):
    """
    Mini-Batch Gradient Descent: usa un GRUPO de datos por paso.

    Parámetros:
        X          : array (N,) datos de entrada (normalizados)
        y          : array (N,) valores reales
        m_init     : float, pendiente inicial
        b_init     : float, intercepto inicial
        lr         : float, learning rate
        epochs     : int, número de épocas
        batch_size : int, tamaño del mini-batch

    Retorna:
        historial : lista de dicts con m, b, mse en cada época
    """
    m = m_init                                  # Pendiente inicial
    b = b_init                                  # Intercepto inicial
    N = len(X)                                  # Número de datos
    historial = []                              # Guardar todo el progreso
    np.random.seed(42)                          # Reproducibilidad

    for epoca in range(epochs):
        # ── Mezclar los datos al inicio de cada época ──
        indices = np.random.permutation(N)

        # ── Recorrer los datos en mini-batches ──
        for inicio in range(0, N, batch_size):
            # Seleccionar el mini-batch actual
            batch_idx = indices[inicio:inicio + batch_size]
            X_batch = X[batch_idx]               # Datos del grupo
            y_batch = y[batch_idx]               # Valores reales del grupo
            n_batch = len(X_batch)               # Tamaño real (puede ser < batch_size al final)

            # ── Predicción para el mini-batch ──
            y_pred_batch = m * X_batch + b

            # ── Error del mini-batch ──
            error_batch = y_pred_batch - y_batch

            # ── Gradientes (promedio del mini-batch) ──
            grad_m = (2 / n_batch) * np.sum(error_batch * X_batch)
            grad_b = (2 / n_batch) * np.sum(error_batch)

            # ── Actualizar parámetros ──
            m = m - lr * grad_m
            b = b - lr * grad_b

        # ── MSE completo al final de la época ──
        mse = calcular_mse(X, y, m, b)
        historial.append({'m': m, 'b': b, 'mse': mse, 'epoca': epoca})

        # ── Imprimir progreso ──
        if epoca % 10 == 0 or epoca == epochs - 1:
            print(f'  Época {epoca:>3}: m={m:>8.3f}  b={b:>8.3f}  MSE={mse:>10.2f}')

    return historial


# ── Ejecutar Mini-Batch GD ──
print('\U0001F4E6\U0001F4E6 MINI-BATCH GRADIENT DESCENT')
print('=' * 55)
print(f'  Parámetros iniciales: m=0, b=0 (mismos que batch y SGD)')
print(f'  Learning rate: 0.05')
print(f'  Épocas: 50')
print(f'  Batch size: 8 eras por grupo')
print()

hist_minibatch = minibatch_gd(
    X, y,
    m_init     = 0,                             # Mismos parámetros iniciales
    b_init     = 0,
    lr         = 0.05,                          # Learning rate intermedio
    epochs     = 50,
    batch_size = 8                              # 8 eras por grupo
)

print(f'\n\u2705 Resultado final:')
print(f'   m = {hist_minibatch[-1]["m"]:.4f}')
print(f'   b = {hist_minibatch[-1]["b"]:.4f}')
print(f'   MSE = {hist_minibatch[-1]["mse"]:.4f}')

## 9. 🔍 Comparación de los 3 tipos

Ahora comparamos los 3 métodos lado a lado:
1. **Trayectorias** en el mapa de contorno cónico del MSE
2. **Curvas de MSE** vs época
3. **Recta final** de cada uno
4. **Tabla resumen** con los resultados numéricos

In [ ]:
# ── Extraer trayectorias de cada método ──
traj = {
    'Batch GD':     {'m': [0]+[h['m'] for h in hist_batch],
                     'b': [0]+[h['b'] for h in hist_batch],
                     'mse': [h['mse'] for h in hist_batch],
                     'color': '#d32f2f', 'marker': 'o'},
    'SGD':          {'m': [0]+[h['m'] for h in hist_sgd],
                     'b': [0]+[h['b'] for h in hist_sgd],
                     'mse': [h['mse'] for h in hist_sgd],
                     'color': '#1565c0', 'marker': 's'},
    'Mini-batch GD':{'m': [0]+[h['m'] for h in hist_minibatch],
                     'b': [0]+[h['b'] for h in hist_minibatch],
                     'mse': [h['mse'] for h in hist_minibatch],
                     'color': '#f57c00', 'marker': '^'},
}

# ── Figura con 3 paneles ──
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# ── Panel 1: Trayectorias en el contorno cónico ──
ax = axes[0]
ax.contourf(Mg, Bg, Zg, levels=20, cmap='viridis', alpha=0.3)
ax.contour(Mg, Bg, Zg, levels=20, cmap='viridis', alpha=0.5)

for nombre, datos in traj.items():
    ax.plot(datos['m'], datos['b'], color=datos['color'],
            marker=datos['marker'], markersize=3, lw=1.5,
            alpha=0.8, label=nombre)
    # Punto final
    ax.plot(datos['m'][-1], datos['b'][-1], color=datos['color'],
            marker='*', markersize=15, zorder=5)

ax.plot(0, 0, 'kx', markersize=12, mew=3, label='Inicio (0,0)', zorder=5)
ax.plot(m_analitico, b_analitico, 'g*', markersize=15, zorder=5,
        label='\u00d3ptimo', alpha=0.7)
ax.set_xlabel('m (pendiente)', fontsize=12)
ax.set_ylabel('b (intercepto)', fontsize=12)
ax.set_title('\U0001F5FA\uFE0F Trayectorias en el cono del error', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)
ax.set_aspect('equal')

# ── Panel 2: Curvas de MSE vs época ──
ax = axes[1]
for nombre, datos in traj.items():
    ax.plot(range(len(datos['mse'])), datos['mse'],
            color=datos['color'], lw=2.5, label=nombre)
ax.set_xlabel('Época', fontsize=12)
ax.set_ylabel('MSE', fontsize=12)
ax.set_title('\U0001F4C9 Convergencia del error', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')                             # Escala logarítmica para ver diferencias

# ── Panel 3: Rectas finales ──
ax = axes[2]
ax.scatter(X, y, c='#4caf50', s=70, edgecolors='#2e7d32',
           lw=1.5, zorder=3, label='Datos reales')
x_line = np.linspace(X.min()-0.5, X.max()+0.5, 100)

for nombre, datos in traj.items():
    m_final = datos['m'][-1]
    b_final = datos['b'][-1]
    ax.plot(x_line, m_final * x_line + b_final,
            color=datos['color'], lw=2.5, label=f'{nombre}')

ax.set_xlabel('Temperatura normalizada', fontsize=12)
ax.set_ylabel('Altura (cm)', fontsize=12)
ax.set_title('\U0001F4CF Rectas finales', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Tabla resumen ──
print('\n' + '=' * 75)
print(f'{"TABLA COMPARATIVA":^75}')
print('=' * 75)
print(f'{"Tipo":<20} {"m final":>10} {"b final":>10} {"MSE final":>12} {"Datos/paso":>12}')
print('-' * 75)
for nombre, datos in traj.items():
    if 'Mini' in nombre:
        datos_paso = '8'
    elif 'SGD' in nombre:
        datos_paso = '1'
    else:
        datos_paso = f'Todos ({N})'
    print(f'{nombre:<20} {datos["m"][-1]:>10.4f} {datos["b"][-1]:>10.4f} {datos["mse"][-1]:>12.4f} {datos_paso:>12}')
print('=' * 75)
print(f'\n\U0001F3AF \u00d3ptimo anal\u00edtico: m={m_analitico:.4f}, b={b_analitico:.4f}, MSE={mse_optimo:.4f}')
print('\n\u2B50 Los 3 métodos convergen a valores similares de m y b')
print('\U0001F440 Pero observa las diferencias en la trayectoria y velocidad')

## 10. 🎯 El efecto del Learning Rate

El **learning rate** ($\eta$) es quizás el hiperparámetro más importante. Controla el **tamaño del paso** al bajar por el cono del error:

| Learning rate | Analogía | Resultado |
|:---:|:---|:---|
| Muy pequeño (0.001) | Pasos de hormiga 🐜 | Converge, pero tarda una eternidad |
| Justo (0.1) | Pasos normales ✅ | Converge rápido y estable |
| Muy grande (1.5) | Saltos enormes 💥 | Se pasa del mínimo, oscila o diverge |

Vamos a comprobarlo experimentalmente con Batch GD.

In [ ]:
# ── Ejecutar Batch GD con 3 learning rates distintos ──
lrs = {
    'lr=0.001 (hormiga \U0001F41C)': {'lr': 0.001, 'color': '#1565c0', 'estilo': '--'},
    'lr=0.1 (justo \u2705)':          {'lr': 0.1,   'color': '#2e7d32', 'estilo': '-'},
    'lr=1.5 (gigante \U0001F4A5)':    {'lr': 1.5,   'color': '#d32f2f', 'estilo': '-.'},
}

resultados_lr = {}
for nombre, config in lrs.items():
    print(f'\n{nombre}')
    hist = batch_gd(X, y, m_init=0, b_init=0, lr=config['lr'], epochs=50)
    resultados_lr[nombre] = {'hist': hist, **config}

# ── Visualización: curvas de MSE ──
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, (nombre, res) in enumerate(resultados_lr.items()):
    ax = axes[i]
    hist = res['hist']

    # Curva de MSE
    mses = [h['mse'] for h in hist]
    ax.plot(range(len(mses)), mses, color=res['color'], lw=2.5)
    ax.set_xlabel('Época', fontsize=12)
    ax.set_ylabel('MSE', fontsize=12)
    ax.set_title(nombre, fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)

    # Límite vertical para ver bien la convergencia
    if res['lr'] <= 0.1:
        ax.set_ylim(bottom=0, top=max(mses[0]*1.1, mses[-1]*2))
    else:
        # Para lr grande, puede diverger
        max_mse = min(max(mses), mses[0] * 5)   # Limitar para que se vea
        ax.set_ylim(bottom=0, top=max_mse)

    # Anotar MSE final
    mse_final = mses[-1]
    ax.annotate(f'MSE final: {mse_final:.1f}',
                xy=(len(mses)-1, mse_final),
                xytext=(len(mses)*0.3, max(mses)*0.7),
                fontsize=10, fontweight='bold', color=res['color'],
                arrowprops=dict(arrowstyle='->', color=res['color']))

plt.suptitle('\U0001F3AF Efecto del Learning Rate en la convergencia',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── Trayectorias en el contorno cónico ──
fig, ax = plt.subplots(figsize=(10, 8))
ax.contourf(Mg, Bg, Zg, levels=20, cmap='viridis', alpha=0.3)
ax.contour(Mg, Bg, Zg, levels=20, cmap='viridis', alpha=0.5)

for nombre, res in resultados_lr.items():
    hist = res['hist']
    ms = [0] + [h['m'] for h in hist]
    bs = [0] + [h['b'] for h in hist]
    # Limitar puntos que se salen mucho del rango
    m_lo, m_hi = m_analitico - rango_m, m_analitico + rango_m
    b_lo, b_hi = b_analitico - rango_b, b_analitico + rango_b
    ms_clip = np.clip(ms, m_lo, m_hi)
    bs_clip = np.clip(bs, b_lo, b_hi)
    ax.plot(ms_clip, bs_clip, color=res['color'], lw=2,
            linestyle=res['estilo'], label=nombre, alpha=0.9)
    ax.plot(ms_clip[-1], bs_clip[-1], color=res['color'],
            marker='*', markersize=15, zorder=5)

ax.plot(0, 0, 'kx', markersize=12, mew=3, label='Inicio', zorder=5)
ax.plot(m_analitico, b_analitico, 'g*', markersize=15, zorder=5,
        label='\u00d3ptimo', alpha=0.7)
ax.set_xlabel('m (pendiente)', fontsize=12)
ax.set_ylabel('b (intercepto)', fontsize=12)
ax.set_title('\U0001F5FA\uFE0F Trayectorias con distintos learning rates',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

# ── Resumen ──
print('\n\U0001F4A1 Conclusiones:')
print('  \U0001F41C lr=0.001: Converge pero muy lento (apenas baja en 50 épocas)')
print('  \u2705 lr=0.1:   Converge rápido y estable (el punto dulce)')
print('  \U0001F4A5 lr=1.5:  Oscila o diverge (los pasos son demasiado grandes)')

## 11. 📝 Resumen

### Tabla resumen de los 3 tipos de descenso de gradiente

| | Batch GD | SGD | Mini-batch GD |
|:---|:---:|:---:|:---:|
| **Datos por paso** | Todos (N) | 1 | batch_size (ej: 8, 32, 64) |
| **Velocidad por paso** | Lenta | Rápida | Media |
| **Estabilidad** | Alta (suave) | Baja (ruidoso) | Media |
| **Convergencia** | Directa al mínimo | Zigzaguea | Suave con algo de ruido |
| **Cuándo usar** | Datos pequeños | Datos enormes / online | **Estándar en Deep Learning** |

### Conexión con Deep Learning

En redes neuronales reales (GPT, ResNet, etc.) se usa **mini-batch SGD** (o variantes como Adam) porque:
- Los datasets tienen millones de imágenes/textos → batch completo es imposible
- El ruido del mini-batch ayuda a **generalizar mejor**
- Se puede paralelizar en GPU (procesar el batch completo de golpe)

### Ideas para experimentar

Prueba modificar este notebook:
- ¿Qué pasa si cambias el **learning rate**? (¿muy pequeño? ¿muy grande?)
- ¿Qué pasa si añades **más ruido** a los datos? (cambia el `std` del ruido gaussiano)
- ¿Qué pasa si usas **más eras** (N=100 o N=1000)?
- ¿Qué pasa si empiezas con **parámetros iniciales** muy lejanos?
- ¿Qué pasa si cambias el **batch_size** del mini-batch? (2, 4, 16, 30)

---

*Notebook creado para el taller de Deep Learning. Solo usa numpy y matplotlib.*